In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np

In [2]:
print(f"PyTorch version: {torch.__version__}")

# Check PyTorch has access to MPS (Metal Performance Shader, Apple's GPU architecture)
print(f"Is MPS (Metal Performance Shader) built? {torch.backends.mps.is_built()}")
print(f"Is MPS available? {torch.backends.mps.is_available()}")

# Set the device      
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

PyTorch version: 2.10.0
Is MPS (Metal Performance Shader) built? True
Is MPS available? True
Using device: mps


In [3]:
# mps_device = torch.device("mps")
mps_device = "mps" if torch.backends.mps.is_available() else "cpu"

transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

trainset = torchvision.datasets.CIFAR10(root='./', train=True,
                                        download=False, transform=transform)    

trainloader = torch.utils.data.DataLoader(trainset, batch_size=4,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./', train=False,
                                       download=False, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=4,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

/Users/bea/Documents/University/Y2/T2/Artificial-Intelligence/venv/lib/python3.13/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [4]:
# # Check that MPS is available
# if not torch.backends.mps.is_available():
#     if not torch.backends.mps.is_built():
#         print("MPS not available because the current PyTorch install was not "
#               "built with MPS enabled.")
#     else:
#         print("MPS not available because the current MacOS version is not 12.3+ "
#               "and/or you do not have an MPS-enabled device on this machine.")

# else:
#     mps_device = torch.device("mps")
#     print("MPS available")

    

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        # New - x to mps
        x = x.to(mps_device) # this entire line is new, yes?
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

net = Net()
net.to(mps_device)


Net(
  (conv1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)

In [5]:
# holder = Net()
# holder

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [7]:
for epoch in range(2):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        no_cuda=True ## DG: added this, that's how it seemed to work, but also need the other additional lines
        # Something around here needs to be also put to mps?
        # Getting new error:
        # RuntimeError: Placeholder storage has not been allocated on MPS device!
        # net = Net()
        # net = net.to(mps_device)
        # Net.x = Net.x.to(mps_device)
        # Because of the syntax with x = x.to syntax I tried something like this
        # inp = inputs.to(mps_device)
        # lab = labels.to(mps_device)
        # inp, lab = data;
        inputs, labels = data
        inputs, labels = inputs.to(mps_device), labels.to(mps_device) # DG added this (w/o no_cuda still getting Placeholder error); source: https://stanford.edu/~shervine/blog/pytorch-how-to-generate-data-parallel/
        
        optimizer.zero_grad()

        # then tried to put these new variables in here:
        # outputs = net(inp)
        # loss = criterion(outputs, lab)
        
        outputs = net(inputs)
        outputs = outputs.to(mps_device) # added this as well
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

       

        # print statistics
        running_loss += loss.item()
        if i % 2000 == 1999:    # print every 2000 mini-batches
            print('[%d, %5d] loss: %.3f' %
                  (epoch + 1, i + 1, running_loss / 2000))
            running_loss = 0.0

print('Finished Training')

[1,  2000] loss: 2.184
[1,  4000] loss: 1.879
[1,  6000] loss: 1.662
[1,  8000] loss: 1.571
[1, 10000] loss: 1.515
[1, 12000] loss: 1.476
[2,  2000] loss: 1.404
[2,  4000] loss: 1.393
[2,  6000] loss: 1.357
[2,  8000] loss: 1.334
[2, 10000] loss: 1.334
[2, 12000] loss: 1.294
Finished Training


Took 1m 45s, so about 40 sec longer than on CPU, which is lame.